# EcoMarket — Atención al Cliente con RAG (Ollama + LangChain + ChromaDB)

**Taller Práctico #2 — Fase 3**

Este notebook implementa el sistema RAG diseñado en las fases 1 y 2 de este taller, sobre el prototipo del Taller 1. El asistente debe poder responder a **cualquier consulta** del cliente o reconocer explícitamente cuándo no tiene información para hacerlo.

## Stack

| Componente | Elección | Decisión documentada en |
|---|---|---|
| **LLM** | Ollama + `llama3.2:3b` (local) | Heredado del Taller 1 |
| **Embeddings** | `intfloat/multilingual-e5-large` (HuggingFace, 1024 dims) | `fase1-componentes.md` |
| **Vector store** | ChromaDB persistente local | `fase1-componentes.md` |
| **Orquestación** | LangChain | `fase1-componentes.md` |
| **Chunking** | Recursivo (500/50) para texto · 1 registro = 1 chunk para JSON | `fase2-base conocimiento.md` |

## Fuentes de conocimiento

| Archivo | Indexado en RAG | Razón |
|---|---|---|
| `data/politicas_devoluciones.md` | Sí | Reglas de negocio, búsqueda semántica útil |
| `data/catalogo_productos.json` | Sí | Productos, precios, devolvibilidad por ítem |
| `data/faq.json` | Sí | Q&A curadas por soporte |
| `data/pedidos.json` | **No** — consulta directa por ID | Datos transaccionales volátiles, búsqueda exacta |

Los prompts viven en archivos planos en `prompts/` para poder iterarlos sin tocar el código:

- `prompts/prompt_pedido.txt` — usado por la herramienta directa de pedidos
- `prompts/prompt_general.txt` — prompt anti-alucinación usado por la cadena RAG
- `prompts/prompt_devolucion.txt` — herencia del Taller 1, conservado como referencia

## Requisitos previos

1. **Python 3.10+** con un entorno virtual activo como kernel del notebook.
2. **Ollama** instalado y corriendo en `http://localhost:11434` (ver sección 2).
3. Ejecutar el notebook **desde la raíz del proyecto** (donde viven `data/` y `prompts/`).
4. Conexión a internet en la primera ejecución: se descargan `llama3.2:3b` (~2 GB) y `intfloat/multilingual-e5-large` (~1.3 GB). Las corridas siguientes leen desde caché y desde `./chroma_db_ecomarket/`.

## 0. Instalación de dependencias

Ejecuta esta celda una sola vez por entorno. Usa `%pip` para garantizar que las librerías se instalen en el mismo Python que usa el kernel del notebook. Las versiones están fijadas en `requirements.txt`.

In [1]:
%pip install -q -r requirements.txt

import warnings
warnings.filterwarnings('ignore')

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Carga de documentos

Cargamos los **cuatro** archivos de `data/` con loaders específicos para cada tipo, aplicando la decisión de la fase 2:

- **Políticas (`.md`)** → un único `Document` con el texto completo, que el splitter recursivo cortará en la sección 4.
- **Catálogo (`.json`)** → un `Document` por producto (chunk atómico), con metadata estructurada (`producto_id`, `categoria`, `devolvible`).
- **FAQ (`.json`)** → un `Document` por par pregunta-respuesta, con metadata `tema`.
- **Pedidos (`.json`)** → **NO** se convierten en `Document`. Se cargan en `PEDIDOS_DICT` indexado por `tracking_number` para consulta determinista por ID (decisión de fase 2: datos volátiles → consulta directa, no búsqueda semántica).

In [2]:
import json
from pathlib import Path
from langchain.schema import Document

DATA_DIR = Path('data')
POLITICAS_PATH = DATA_DIR / 'politicas_devoluciones.md'
CATALOGO_PATH = DATA_DIR / 'catalogo_productos.json'
FAQ_PATH = DATA_DIR / 'faq.json'
PEDIDOS_PATH = DATA_DIR / 'pedidos.json'


def cargar_politicas(path: Path) -> list[Document]:
    texto = path.read_text(encoding='utf-8')
    return [Document(
        page_content=texto,
        metadata={'source': 'politicas_devoluciones', 'tipo': 'politica'},
    )]


def cargar_catalogo(path: Path) -> list[Document]:
    productos = json.loads(path.read_text(encoding='utf-8'))
    docs = []
    for p in productos:
        contenido = (
            f"Producto: {p['nombre']}\n"
            f"Categoría: {p.get('categoria', 'N/A')}\n"
            f"Precio: {p.get('precio', 'N/A')}\n"
            f"Material: {p.get('material', 'N/A')}\n"
            f"Admite devolución: {p.get('devolvible', 'N/A')}\n"
            f"Stock: {p.get('stock', 'N/A')}\n"
            f"Descripción: {p.get('descripcion', '')}"
        )
        docs.append(Document(
            page_content=contenido,
            metadata={
                'source': 'catalogo',
                'tipo': 'producto',
                'producto_id': p.get('producto_id', ''),
                'categoria': p.get('categoria', ''),
                'devolvible': str(p.get('devolvible', '')),
            },
        ))
    return docs


def cargar_faq(path: Path) -> list[Document]:
    pares = json.loads(path.read_text(encoding='utf-8'))
    docs = []
    for q in pares:
        contenido = f"Pregunta: {q['pregunta']}\nRespuesta: {q['respuesta']}"
        docs.append(Document(
            page_content=contenido,
            metadata={
                'source': 'faq',
                'tipo': 'faq',
                'tema': q.get('tema', ''),
            },
        ))
    return docs


def cargar_pedidos(path: Path) -> dict[str, dict]:
    pedidos = json.loads(path.read_text(encoding='utf-8'))
    return {str(p['tracking_number']): p for p in pedidos}


docs_politicas = cargar_politicas(POLITICAS_PATH)
docs_catalogo = cargar_catalogo(CATALOGO_PATH)
docs_faq = cargar_faq(FAQ_PATH)
PEDIDOS_DICT = cargar_pedidos(PEDIDOS_PATH)

print(f"Políticas (Documents): {len(docs_politicas)}")
print(f"Productos del catálogo (Documents): {len(docs_catalogo)}")
print(f"FAQ (Documents): {len(docs_faq)}")
print(f"Pedidos (NO indexados, consulta directa): {len(PEDIDOS_DICT)}")

Políticas (Documents): 1
Productos del catálogo (Documents): 15
FAQ (Documents): 15
Pedidos (NO indexados, consulta directa): 10


## 2. Instalación y arranque de Ollama (Windows)

Ollama se instala **fuera del notebook**, una sola vez:

1. Descarga el instalador desde https://ollama.com/download/windows y ejecútalo.
2. Tras instalar, Ollama queda corriendo como servicio en `http://localhost:11434`. Si no lo está, abre una terminal y ejecuta `ollama serve`.
3. Verifica en una terminal nueva: `ollama --version`.

> En Linux/macOS puedes usar `curl -fsSL https://ollama.com/install.sh | sh`. En Windows el instalador oficial es la forma soportada.

La siguiente celda descarga el modelo `llama3.2:3b` (~2 GB la primera vez). Si ya lo descargaste antes, Ollama reportará que el modelo ya existe.

In [3]:
!ollama pull llama3.2:3b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


## 3. LLM con LangChain + Ollama

Conectamos LangChain al modelo servido por Ollama. `validate_model_on_init=True` falla rápido si Ollama no está corriendo o si el modelo no fue descargado.

In [4]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2:3b", validate_model_on_init=True)

llm.invoke("Responde brevemente: ¿qué es EcoMarket?").content

'EcoMarket es una plataforma de comercio electrónico en línea que se centra en la venta y el trueque de productos ecológicos, sostenibles y respetuosos con el medio ambiente. Ofrece una variedad de artesanías, productos naturales, ropa reciclada y más, impulsando prácticas de consumo responsable y sostenible.'

## 4. Construcción del vectorstore (ChromaDB persistente)

Aplicamos exactamente el pipeline de la fase 2 (Load → Split → Embed → Store → Persist):

- **Embeddings**: `intfloat/multilingual-e5-large` cargado vía `HuggingFaceEmbeddings`. Se descarga (~1.3 GB) en la primera ejecución y se cachea en `~/.cache/huggingface`.
- **Splitting diferenciado por tipo**:
  - Las **políticas** (texto libre en Markdown) se segmentan con `RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)`. El splitter respeta la jerarquía `\n\n → \n → . → espacio`, manteniendo cláusulas completas.
  - El **catálogo** y la **FAQ** ya son atómicos (1 registro = 1 chunk). No se re-segmentan: dividirlos diluye la unidad de significado y borra la metadata estructurada.
- **Persistencia**: `Chroma.from_documents(..., persist_directory=...)` guarda el índice en `./chroma_db_ecomarket/`. La próxima vez que se ejecute la celda, se cargará desde disco sin reprocesar embeddings.
- **Retriever**: `k=4` para que el LLM reciba contexto suficiente sin saturar la ventana de contexto.

In [5]:
import os
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter

embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large")

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks_politicas = splitter.split_documents(docs_politicas)
chunks_estructurados = docs_catalogo + docs_faq
todos_los_chunks = chunks_politicas + chunks_estructurados

print(f"Chunks de políticas (recursivo): {len(chunks_politicas)}")
print(f"Chunks de catálogo + FAQ (atómicos): {len(chunks_estructurados)}")
print(f"Total de chunks a indexar: {len(todos_los_chunks)}")

PERSIST_DIR = './chroma_db_ecomarket'

if os.path.exists(PERSIST_DIR) and os.listdir(PERSIST_DIR):
    print(f"\nCargando vectorstore existente desde {PERSIST_DIR} ...")
    vectorstore = Chroma(persist_directory=PERSIST_DIR, embedding_function=embeddings)
else:
    print(f"\nCreando vectorstore nuevo en {PERSIST_DIR} ...")
    vectorstore = Chroma.from_documents(
        documents=todos_los_chunks,
        embedding=embeddings,
        persist_directory=PERSIST_DIR,
    )

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Retriever listo (top-k=4).")

Chunks de políticas (recursivo): 8
Chunks de catálogo + FAQ (atómicos): 30
Total de chunks a indexar: 38

Creando vectorstore nuevo en ./chroma_db_ecomarket ...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Retriever listo (top-k=4).


## 5. Carga de prompts

Los prompts se mantienen en archivos planos en `prompts/` para poder iterarlos sin tocar el código del notebook. Cargamos **dos** que se usan en el flujo principal:

- `prompt_pedido.txt` — usado por la **herramienta directa de pedidos** (sección 6) cuando la consulta trae un tracking number.
- `prompt_general.txt` — usado por la **cadena RAG** (sección 7) para todas las demás consultas. Incluye la instrucción anti-alucinación que dispara el fallback "No tengo información…" cuando el contexto recuperado no es suficiente.

(`prompt_devolucion.txt` queda en el repo como referencia del diseño original del Taller 1.)

In [6]:
from langchain.prompts import PromptTemplate

PROMPTS_DIR = Path('prompts')

prompt_pedido_tpl = (PROMPTS_DIR / 'prompt_pedido.txt').read_text(encoding='utf-8')
prompt_general_tpl = (PROMPTS_DIR / 'prompt_general.txt').read_text(encoding='utf-8')

prompt_pedido = PromptTemplate.from_template(prompt_pedido_tpl)
prompt_general = PromptTemplate.from_template(prompt_general_tpl)

print('--- prompt_pedido.txt ---')
print(prompt_pedido_tpl)
print('--- prompt_general.txt ---')
print(prompt_general_tpl)

--- prompt_pedido.txt ---
Actúa como un agente experto de servicio al cliente de EcoMarket, especializado en pedidos y logística.

### Contexto del sistema:
Tienes acceso a la siguiente información de pedidos:
{data}

### Tarea:
El cliente quiere conocer el estado de su pedido con número: {tracking_number}

### Instrucciones:
- Responde únicamente con base en la información proporcionada.
- NO inventes datos ni hagas suposiciones.
- Si el pedido no existe, indícalo claramente.
- Usa un tono amable, profesional y cercano.
- Incluye:
  1. Estado del pedido
  2. Fecha estimada de entrega
  3. Explicación breve del estado
- Si el pedido está retrasado:
  - Ofrece disculpas
  - Explica brevemente la causa
- Mantén la respuesta clara y breve (máximo 5 líneas).

### Formato de respuesta esperado:
Saludo breve
Estado del pedido
Fecha de entrega
Mensaje adicional (si aplica)

### Respuesta:

--- prompt_general.txt ---
Actúa como un agente de servicio al cliente de EcoMarket, una empresa de prod

## 6. Herramienta directa de pedidos (no RAG)

La consulta *"¿dónde está mi pedido 1003?"* no se resuelve por similitud semántica: dos IDs numéricos pueden tener embeddings parecidos sin significar lo mismo. Por eso la fase 2 decide **excluir `pedidos.json` del vectorstore** y exponerlo como una herramienta determinista.

Esta función consulta `PEDIDOS_DICT` (cargado en la sección 1) por `tracking_number` exacto y devuelve un texto formateado, o `None` si el ID no existe.

In [7]:
def consultar_pedido(tracking_number: str) -> str | None:
    pedido = PEDIDOS_DICT.get(str(tracking_number))
    if not pedido:
        return None
    return (
        f"Pedido {pedido['tracking_number']}\n"
        f"Estado: {pedido['estado']}\n"
        f"Fecha estimada de entrega: {pedido['fecha_entrega']}\n"
        f"Detalle: {pedido['detalle']}"
    )

print(consultar_pedido('1003'))
print('---')
print(consultar_pedido('9999'))

Pedido 1003
Estado: Retrasado
Fecha estimada de entrega: 2026-04-18
Detalle: Retraso por alta demanda logística.
---
None


## 7. Pipeline RAG con router simple

El flujo completo es:

1. **Router** (regex): si la pregunta contiene un número de tracking de 4 dígitos válido (1001-1010 en este dataset), se invoca la **herramienta directa de pedidos** + `prompt_pedido`.
2. **Cadena RAG única** (todo lo demás): `retriever.invoke(pregunta)` recupera los 4 chunks más similares de la base unificada (políticas + catálogo + FAQ), se concatenan en `{contexto}` y se inyectan al `prompt_general`. El prompt incluye la instrucción anti-alucinación, así que el **fallback** "No tengo información…" se dispara automáticamente cuando los chunks recuperados no cubren la pregunta. **No es una rama de código separada**: es comportamiento del prompt.

Si llega un tracking number que no existe (ej. 9999), la rama 1 cae al RAG general como degradación natural.

In [8]:
import re


def extraer_tracking(pregunta: str) -> str | None:
    m = re.search(r'\b(\d{4})\b', pregunta)
    return m.group(1) if m else None


def responder(pregunta: str) -> dict:
    # Rama 1 — tool directa de pedidos
    tracking = extraer_tracking(pregunta)
    if tracking:
        info_pedido = consultar_pedido(tracking)
        if info_pedido:
            prompt_final = prompt_pedido.format(
                data=info_pedido,
                tracking_number=tracking,
            )
            return {
                'answer': llm.invoke(prompt_final).content,
                'fuentes': [{'source': 'pedidos', 'tracking_number': tracking}],
                'ruta': 'tool_pedidos',
            }

    # Rama 2 — cadena RAG única con prompt anti-alucinación
    docs_recuperados = retriever.invoke(pregunta)
    contexto = "\n\n".join(d.page_content for d in docs_recuperados)
    prompt_final = prompt_general.format(contexto=contexto, pregunta=pregunta)
    return {
        'answer': llm.invoke(prompt_final).content,
        'fuentes': [d.metadata for d in docs_recuperados],
        'ruta': 'rag',
    }

## 8. Helper de impresión

Imprime la respuesta del modelo y las fuentes que usó (qué documento o pedido) para que cada respuesta sea **trazable**.

In [9]:
def imprimir_respuesta(resultado: dict) -> None:
    print(f"[Ruta: {resultado['ruta']}]\n")
    print(resultado['answer'])
    print("\nFuentes:")
    for i, meta in enumerate(resultado['fuentes'], 1):
        src = meta.get('source', 'desconocido')
        tipo = meta.get('tipo', '')
        extra = (
            meta.get('producto_id')
            or meta.get('tema')
            or meta.get('tracking_number')
            or ''
        )
        etiqueta = f"{src}" + (f" [{tipo}]" if tipo else "") + (f" #{extra}" if extra else "")
        print(f"  - [{i}] {etiqueta}")

## 9. Pruebas

Cuatro escenarios diseñados para validar las decisiones del taller y demostrar que el modelo cambia de comportamiento según el documento recuperado y el prompt aplicado:

| # | Pregunta | Ruta esperada | Qué evidencia |
|---|---|---|---|
| 1 | Estado del pedido 1003 | `tool_pedidos` | El router detecta tracking, la herramienta consulta `pedidos.json` por ID exacto, no usa el vectorstore. |
| 2 | ¿Puedo devolver un cepillo de dientes de bambú? | `rag` | El retriever recupera política + producto del catálogo, el LLM concluye **no devolvible** citando el contexto. |
| 3 | ¿Tienen botellas térmicas y cuánto cuestan? | `rag` | Recupera entrada del catálogo, responde con el precio real del JSON. |
| 4 | ¿Cuál es la capital de Francia? | `rag` | Fuera de dominio: ningún chunk relevante, el `prompt_general` dispara el fallback "No tengo información…". |

In [10]:
imprimir_respuesta(responder("¿Cuál es el estado de mi pedido 1003?"))

[Ruta: tool_pedidos]

Hola, gracias por contactarnos. 

Esto es un recordatorio sobre el estado de su pedido con número: 1003.

El estado actual del pedido es "Retrasado".
La fecha estimada de entrega es: 2026-04-18.
Está retrasado debido a alta demanda logística, nos disculpamos por cualquier inconveniente que esto pueda causarle.

Fuentes:
  - [1] pedidos #1003


In [11]:
imprimir_respuesta(responder("¿Puedo devolver un cepillo de dientes de bambú?"))

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[Ruta: rag]

No tengo información sobre eso en los documentos de EcoMarket. Te recomiendo contactar a soporte humano en soporte@ecomarket.com.

Fuentes:
  - [1] catalogo [producto] #P001
  - [2] politicas_devoluciones [politica]
  - [3] faq [faq] #devoluciones
  - [4] politicas_devoluciones [politica]


In [12]:
imprimir_respuesta(responder("¿Tienen botellas térmicas y cuánto cuestan?"))

[Ruta: rag]

Sí, tenemos botellas térmicas disponibles. La botella térmica de acero inoxidable 750ml cuesta $85,000. Ten en cuenta que tenemos una garantía de 6 meses contra defectos de fabricación. (Fuente: política de garantía)

Fuentes:
  - [1] catalogo [producto] #P002
  - [2] politicas_devoluciones [politica]
  - [3] faq [faq] #garantías
  - [4] faq [faq] #tienda


In [13]:
imprimir_respuesta(responder("¿Cuál es la capital de Francia?"))

[Ruta: rag]

Lo siento, pero no tengo información sobre eso en los documentos de EcoMarket. Te recomiendo contactar a soporte humano en soporte@ecomarket.com.

Fuentes:
  - [1] faq [faq] #pagos
  - [2] faq [faq] #tienda
  - [3] faq [faq] #envíos
  - [4] faq [faq] #envíos


## 10. Conclusiones, limitaciones y suposiciones

### Lo que demostró el sistema

- **Coherencia entre fases**: las decisiones de fase 1 (e5-large + ChromaDB) y fase 2 (chunking diferenciado, pedidos fuera del vectorstore) se materializan en el código sin atajos.
- **Cobertura del enunciado del Taller 2**: el asistente responde a cualquier consulta o reconoce explícitamente cuándo no tiene herramientas (test 4).
- **Trazabilidad**: cada respuesta incluye sus fuentes, lo que permite auditar y mitiga el riesgo ético de alucinaciones identificado en el Taller 1.

### Limitaciones reconocidas

- **Router por regex**: detecta cualquier número de 4 dígitos como posible tracking. Si un cliente pregunta por un código postal de 4 dígitos, el router lo intentará primero como pedido. La degradación es benigna (cae al RAG si el ID no existe), pero un router más maduro usaría intent classification.
- **Sin filtrado por metadata en el retriever**: el `as_retriever` actual hace top-k global. Para producción se podría usar `search_kwargs={"filter": {"tipo": "producto"}}` para restringir consultas de catálogo a productos.
- **Modelo pequeño**: `llama3.2:3b` es ágil pero ocasionalmente no respeta el límite de 6 líneas o el formato exacto del fallback. Un modelo más grande (Llama 3.1 8B, Mistral 7B) reduciría esa varianza.
- **Sin evaluación cuantitativa**: no se mide precisión ni recall del retrieval. Una mejora natural sería montar un set de preguntas etiquetadas y medir hit rate.

### Suposiciones de ejecución

- **Hardware**: ≥8 GB RAM, GPU opcional (acelera embeddings; el LLM corre en CPU sin problemas en `llama3.2:3b`).
- **Conectividad**: requerida solo en la primera ejecución (descarga de modelo y embeddings). Después es 100% local.
- **Datos**: `pedidos.json` se asume sincronizado al momento de la consulta; en producción esta tool consultaría una API o base relacional.
- **Idioma**: consultas y documentos en español. Otros idiomas funcionarían (e5 es multilingüe) pero no se han validado.